# Week 1 Assignment GitHub, Pandas First-Look Protocol & EDA
**Dataset:** Titanic Passenger Data (Kaggle) &nbsp;|&nbsp; **Domain:** History
**Author:** [Your Name] &nbsp;|&nbsp; **Mentor:** Laiba Sattar

---
## Section 1 Dataset Introduction

I chose the **Titanic Passenger Data** dataset because it is a clean, well-documented, real-world dataset that is small enough to explore quickly but rich enough to practice every command in this assignment: it has numeric columns (`Age`, `Fare`), categorical columns (`Sex`, `Embarked`, `Pclass`), a binary target (`Survived`), free text (`Name`), and genuine missing data (`Age`, `Cabin`, `Embarked`) which makes it a realistic sandbox for the First-Look Protocol and deeper EDA.

- **Rows / Columns:** 891 rows × 12 columns (meets the 500-row / 5-column minimum)
- **Source:** kaggle.com/c/titanic/data
- **Why it's interesting:** it lets us test whether survival was driven by class, gender, age, or fare a question with a real historical answer we can check our findings against.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('titanic.csv')
df.head()

## Section 2 The 7-Command First-Look Protocol

### C1. `df.shape` dataset dimensions

In [ ]:
df.shape

(891, 12)

**Interpretation:** The dataset has **891 rows and 12 columns**. That's enough data for solid exploratory analysis and classical ML (logistic regression, decision trees), though it would be small for a deep learning model.

### C2. `df.dtypes` data type of every column

In [ ]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

**Interpretation:** Most types look correct `Survived`, `Pclass`, `SibSp`, `Parch` are integers, `Age` and `Fare` are floats. `Age` is `float64` rather than `int` because it contains missing values (NaN forces a float type). `Name`, `Sex`, `Ticket`, `Cabin`, and `Embarked` are text (object/string) columns. `Ticket` mixes letters and numbers, so it will need extra investigation if used as a feature.

### C3. `df.info()` non-null counts, dtypes, memory usage

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

**Interpretation:** `Age` has 714 non-null values → **177 missing (19.9%)**. `Cabin` has only 204 non-null → **687 missing (77.1%)**, nearly useless as-is. `Embarked` is missing just 2 rows (0.2%) negligible. These three columns each need a different fix before modelling: drop/rebuild `Cabin`, impute `Age`, and fill `Embarked` with its mode.

### C4. `df.describe()` summary statistics for numeric columns

In [ ]:
df.describe()

       PassengerId  Survived  Pclass     Age   SibSp   Parch    Fare
count       891.00    891.00  891.00  714.00  891.00  891.00  891.00
mean        446.00      0.38    2.31   29.70    0.52    0.38   32.20
std         257.35      0.49    0.84   14.53    1.10    0.81   49.69
min           1.00      0.00    1.00    0.42    0.00    0.00    0.00
25%         223.50      0.00    2.00   20.12    0.00    0.00    7.91
50%         446.00      0.00    3.00   28.00    0.00    0.00   14.45
75%         668.50      1.00    3.00   38.00    1.00    0.00   31.00
max         891.00      1.00    3.00   80.00    8.00    6.00  512.33

**Interpretation:** `Fare`'s mean (**32.20**) is much higher than its median (**14.45**) a classic sign of a right-skewed distribution driven by a handful of expensive first-class tickets. The **£0 minimum** fare and **£512 maximum** fare (about 35× the median) both warrant investigation, free tickets were likely crew/staff, and the £512 fares belong to a small group of first-class passengers sharing one ticket.

### C5. `df.isnull().sum()` exact missing-value count per column

In [ ]:
df.isnull().sum()
# Percentage missing
(df.isnull().sum() / len(df) * 100).round(1)

Age         177
Cabin       687
Embarked      2
(all other columns: 0)

Percentage missing:
Age         19.9
Cabin       77.1
Embarked     0.2

**Interpretation:** Confirms C3 `Cabin` (77% missing) should probably be dropped or converted into a binary 'has cabin recorded' flag rather than imputed. `Age` (20% missing) is worth imputing (e.g., median per `Pclass`/`Sex` group) rather than dropping, since it's a strong predictor. `Embarked`'s 2 missing rows can simply be filled with the mode (`'S'`).

### C6. `df['Survived'].value_counts()` class balance of the target

In [ ]:
df['Survived'].value_counts()
df['Survived'].value_counts(normalize=True) * 100

0    549   (61.6%)
1    342   (38.4%)

**Interpretation:** Only **38.4%** of passengers survived this is a moderately **imbalanced target**. A model that blindly predicts 'no one survived' would score 61.6% accuracy while being useless. This means accuracy alone is a misleading metric here; we'd want **F1-score or ROC-AUC** when evaluating any survival-prediction model.

### C7. `df.duplicated().sum()` exact duplicate rows

In [ ]:
df.duplicated().sum()

0

**Interpretation:** No duplicate rows in this dataset. That's expected for Titanic (each `PassengerId` is unique), but in messier datasets (e.g., e-commerce transaction logs) this check often turns up hundreds of duplicates that silently inflate totals and skew groupby results — so it's a habit worth keeping regardless of the dataset.

## Section 3 — 20+ Additional EDA Commands
At least 15 of the commands below are taken directly from the assignment's list of 15; the remainder explore further into the Pandas library (`skew`, `kurt`, `cov`, `mode`, `memory_usage`).

### 1. `df.head(10)` — first 10 rows

In [ ]:
df.head(10)

   PassengerId  Survived  ...     Sex   Age
0            1         0  ...    male  22.0
1            2         1  ...  female  38.0
2            3         1  ...  female  26.0
3            4         1  ...  female  35.0
4            5         0  ...    male  35.0
5            6         0  ...    male   NaN
6            7         0  ...    male  54.0
7            8         0  ...    male   2.0
8            9         1  ...  female  27.0
9           10         1  ...  female  14.0

[10 rows x 6 columns]

**Interpretation:** The first 10 rows look well-formatted — no obvious symbol/encoding issues. Names follow a consistent `Last, Title. First` pattern, which we can exploit later to extract titles (Mr., Mrs., Miss., Master.).

### 2. `df.tail(5)` — last 5 rows

In [ ]:
df.tail(5)

     PassengerId  Survived  Pclass   Fare Embarked
886          887         0       2  13.00        S
887          888         1       1  30.00        S
888          889         0       3  23.45        S
889          890         1       1  30.00        C
890          891         0       3   7.75        Q

**Interpretation:** No formatting problems at the end of the file either — sometimes data-ingestion errors (truncated rows, shifted columns) collect at the tail, but that's not the case here.

### 3. `df.sample(15, random_state=42)` — 15 random rows

In [ ]:
df.sample(15, random_state=42)

(15 random rows shown — see notebook output)

**Interpretation:** A random sample confirms the head/tail impression holds throughout the dataset — no obvious block of corrupted rows in the middle that head()/tail() alone would miss.

### 4. `df.columns.tolist()` — all column names

In [ ]:
df.columns.tolist()

['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

**Interpretation:** All column names are clean — no leading/trailing spaces or inconsistent casing (which would silently break `df['Age']` vs `df['Age ']` lookups).

### 5. `df.nunique()` — unique value count per column

In [ ]:
df.nunique()

PassengerId    891
Survived         2
Pclass           3
Name           891
Sex              2
Age             88
SibSp            7
Parch            7
Ticket         681
Fare           248
Cabin          147
Embarked         3
dtype: int64

**Interpretation:** `PassengerId` and `Name` have 891 unique values each (as expected — they're identifiers). `Sex` has only 2 and `Pclass` only 3 — confirming they're true categorical columns. `Ticket` has 681 unique values across 891 rows, meaning some tickets are shared by multiple passengers (families/groups travelling together).

### 6. `df['Sex'].unique()` / `df['Embarked'].unique()` — distinct category values

In [ ]:
df['Sex'].unique()
df['Embarked'].unique()

Sex: ['male' 'female']
Embarked: ['S' 'C' 'Q' nan]

**Interpretation:** No inconsistent labelling (e.g., no stray `'Male'` vs `'male'`). `Embarked` has 3 real ports — S (Southampton), C (Cherbourg), Q (Queenstown) — plus the 2 missing values we already flagged.

### 7. `df.corr(numeric_only=True)` — correlation matrix

In [ ]:
df.corr(numeric_only=True).round(2)

             PassengerId  Survived  Pclass   Age  SibSp  Parch  Fare
PassengerId         1.00     -0.01   -0.04  0.04  -0.06  -0.00  0.01
Survived           -0.01      1.00   -0.34 -0.08  -0.04   0.08  0.26
Pclass             -0.04     -0.34    1.00 -0.37   0.08   0.02 -0.55
Age                 0.04     -0.08   -0.37  1.00  -0.31  -0.19  0.10
SibSp              -0.06     -0.04    0.08 -0.31   1.00   0.41  0.16
Parch              -0.00      0.08    0.02 -0.19   0.41   1.00  0.22
Fare                0.01      0.26   -0.55  0.10   0.16   0.22  1.00

**Interpretation:** `Survived` correlates most strongly (negatively) with `Pclass` (**-0.34**) and positively with `Fare` (**0.26**) — richer, higher-class passengers were more likely to survive. `Pclass` and `Fare` are strongly negatively correlated (**-0.55**), which makes sense since `Pclass` 1 = most expensive.

### 8. `df['Age'].hist(bins=20)` — distribution shape

In [ ]:
df['Age'].plot(kind='hist', bins=20, title='Age Distribution')

(histogram — right-skewed, peak around 20-30, small bump near infancy)

**Interpretation:** Age is **right-skewed** with a peak in the 20s-30s and a small secondary bump for infants/young children. There's a long thin tail out to 80. This confirms the mean (29.7) sits slightly above the median (28).

### 9. `df.groupby('Pclass').mean(numeric_only=True)` — averages by class

In [ ]:
df.groupby('Pclass').mean(numeric_only=True).round(2)

        PassengerId  Survived    Age  SibSp  Parch   Fare
Pclass                                                   
1            461.60      0.63  38.23   0.42   0.36  84.15
2            445.96      0.47  29.88   0.40   0.38  20.66
3            439.15      0.24  25.14   0.62   0.39  13.68

**Interpretation:** Survival rate drops sharply by class: **63%** in 1st class, **47%** in 2nd, only **24%** in 3rd. Average fare also drops from **£84** to **£14**. Class was clearly one of the strongest survival factors — likely tied to cabin location and access to lifeboats.

### 10. `df['Pclass'].value_counts(normalize=True)` — class proportions

In [ ]:
(df['Pclass'].value_counts(normalize=True) * 100).round(1)

3    55.1%\n1    24.2%\n2    20.7%

**Interpretation:** Just over half the passengers (55%) travelled 3rd class — combined with their lower survival rate, this is a big part of why the overall survival rate is only 38%.

### 11. `df.select_dtypes(include='object')` — all text/categorical columns

In [ ]:
df.select_dtypes(include='object').columns.tolist()

['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']

**Interpretation:** Five text columns exist. `Name` and `Ticket` are effectively free-text/ID fields that need feature engineering (e.g., extracting titles) before they're useful in a model; `Sex`, `Cabin`, `Embarked` are true categoricals.

### 12. `df['Name'].str.contains('Dr.').sum()` — search text columns

In [ ]:
df['Name'].str.contains('Dr.', regex=False).sum()
df['Name'].str.contains('Master.', regex=False).sum()

Dr. count: 7\nMaster count: 40

**Interpretation:** 7 passengers held the title 'Dr.' and 40 boys were recorded with the title 'Master.' (the historical convention for young boys). This shows titles embedded in `Name` are a rich, extractable feature for age/social-status estimation.

### 13. `df.sort_values('Fare', ascending=False).head(5)` — top 5 fares

In [ ]:
df.sort_values('Fare', ascending=False).head(5)[['Name','Pclass','Fare']]

                                   Name  Pclass      Fare
737              Lesurer, Mr. Gustave J       1  512.3292
679  Cardeza, Mr. Thomas Drake Martinez       1  512.3292
258                    Ward, Miss. Anna       1  512.3292
438                   Fortune, Mr. Mark       1  263.0000
88           Fortune, Miss. Mabel Helen       1  263.0000

**Interpretation:** The three highest fares (£512.33 each) belong to first-class passengers who shared a single ticket — explaining why `Fare` looked like such an extreme outlier in `describe()`. This is a per-ticket group fare, not necessarily a per-person price.

### 14. `df[df['Age'] > 60]` — filter rows by condition

In [ ]:
over_60 = df[df['Age'] > 60]
len(over_60)

22

**Interpretation:** Only 22 passengers were over 60. Scanning their survival column shows most did **not** survive — consistent with the 'women and children first' evacuation policy working against elderly men in particular.

### 15. `pd.pivot_table(df, values='Survived', index='Pclass', columns='Sex', aggfunc='mean')` — cross-tab

In [ ]:
pd.pivot_table(df, values='Survived', index='Pclass', columns='Sex', aggfunc='mean').round(2)

Sex     female  male
Pclass              
1         0.97  0.37
2         0.92  0.16
3         0.50  0.14

**Interpretation:** This is the single most revealing table in the dataset. **97% of 1st-class women survived** versus **14% of 3rd-class men** — a massive gap. Gender mattered even more than class: even 3rd-class women (50%) outsurvived 1st-class men (37%).

### 16. `df.skew(numeric_only=True)` — skewness of numeric columns

In [ ]:
df.skew(numeric_only=True).round(2)

PassengerId    0.00
Survived       0.48
Pclass        -0.63
Age            0.39
SibSp          3.70
Parch          2.75
Fare           4.79
dtype: float64

**Interpretation:** `Fare` is heavily right-skewed (**4.79**) and `SibSp`/`Parch` are also skewed (**3.70 / 2.75**) since most passengers travelled alone or with 1 relative, but a few large families pull the tail out. `Age` is only mildly skewed (**0.39**), roughly a bell shape.

### 17. `df.kurt(numeric_only=True)` — kurtosis (tail heaviness)

In [ ]:
df.kurt(numeric_only=True).round(2)

PassengerId    -1.20
Survived       -1.78
Pclass         -1.28
Age             0.18
SibSp          17.88
Parch           9.78
Fare           33.40
dtype: float64

**Interpretation:** `Fare` has very high kurtosis (**33.4**), confirming extreme outliers (those £512 tickets) far from the bulk of the distribution — matching what C4 and command 13 already showed.

### 18. `df[['Age','Fare']].cov()` — covariance

In [ ]:
df[['Age','Fare']].cov().round(2)

         Age     Fare
Age   211.02    73.85
Fare   73.85  2469.44

**Interpretation:** Positive covariance (**73.85**) between Age and Fare — older passengers tended to pay somewhat higher fares, consistent with older travellers more often being in 1st class. Covariance's scale-dependence is why the correlation coefficient (0.10, from command 7) is the more interpretable number.

### 19. `df['Embarked'].mode()` — most common category

In [ ]:
df['Embarked'].mode()

'S'  (Southampton)

**Interpretation:** Southampton is the most common embarkation port — this is exactly the value we'll use to fill the 2 missing `Embarked` rows identified back in C5.

### 20. `df.memory_usage(deep=True)` — memory footprint per column

In [ ]:
df.memory_usage(deep=True)

Index            132
PassengerId     7128
Survived        7128
Pclass          7128
Name           67685
Sex            47851
Age             7128
SibSp           7128
Parch           7128
Ticket         49674
Fare            7128
Cabin          32712
Embarked       44514
dtype: int64

**Interpretation:** Text columns (`Name`, `Ticket`, `Embarked`, `Sex`) take up disproportionately more memory per column than the numeric columns, since Python strings are stored as full objects rather than fixed-width numbers. Total footprint is tiny (under 300 KB) — no memory concerns for a dataset this size.

### 21. `df.groupby('Sex')['Survived'].mean()` — survival rate by gender

In [ ]:
df.groupby('Sex')['Survived'].mean().round(3)

female    0.742\nmale      0.189

**Interpretation:** Women survived at nearly **4× the rate of men** (74.2% vs 18.9%) — the single strongest signal in the whole dataset, stronger even than `Pclass`. This is the clearest quantitative evidence of the 'women and children first' protocol.

### 22. `df['Fare'].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99])` — fare distribution at key percentiles

In [ ]:
df['Fare'].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).round(2)

0.10      7.55
0.25      7.91
0.50     14.45
0.75     31.00
0.90     77.96
0.99    249.01
Name: Fare, dtype: float64

**Interpretation:** The jump from the 90th percentile (£77.96) to the 99th percentile (£249.01) is enormous compared to the jump from 10th to 90th — hard confirmation that a small number of very expensive tickets are dragging the mean fare well above the median.

## Section 4 — Three Key Findings (Plain English)

1. **Gender was the single biggest survival factor.** Women survived at nearly 4 times the rate of men (74% vs 19%) — far outweighing any other variable in the dataset. This lines up with the historical 'women and children first' evacuation policy.

2. **Ticket class amplified the gender effect, and money bought survival odds.** Within each class, women out-survived men by a huge margin, but class still mattered a lot on its own: 1st-class passengers survived at 63% versus just 24% in 3rd class. The pivot table (command 15) shows the two effects stack — a 1st-class woman had a 97% survival rate, while a 3rd-class man had only 14%.

3. **Missing data isn't random and needs a deliberate strategy per column.** `Cabin` (77% missing) is too sparse to impute reliably and is better turned into a binary 'cabin recorded' flag; `Age` (20% missing) is worth imputing carefully (e.g., median by `Pclass`/`Sex` group) since C3–C5 showed it's linked to both class and fare; `Embarked` (0.2% missing) can just be filled with its mode ('S'). Blindly dropping all rows with any missing value would have thrown away nearly 80% of the dataset because of `Cabin` alone.
